# Question 1: Algorithm Design for University Registration System

**Student:** Victor Isingizwe Munezero &nbsp;|&nbsp; **Reg. No:** 26015486 &nbsp;|&nbsp; **Course:** Advanced System Analysis and Design of Algorithms &nbsp;|&nbsp; **Lecturer:** Dr. Jean Bosco Musabe &nbsp;|&nbsp; **Date:** June 20, 2026

## Introduction

University registration systems are core information systems that manage student enrollment, course allocation, and academic records. A reliable registration algorithm must guarantee **correctness** (no duplicate registrations), **completeness** (no missing data), and **resource constraint enforcement** (course capacity limits). This notebook designs, implements, and **tests** such a system end-to-end.

## (a) Python Function-Based Algorithm for Student Registration

The registration function accepts student details and stores them only if every validation rule passes. A function-based (procedural) design is used, as required by the question, with a module-level dictionary acting as the student store for O(1) average-case lookup.

In [1]:
students = {}

def register_student(student_id, name, email):
    """Register a new student. Returns (success: bool, message: str)."""
    if not student_id or not name or not email:
        return False, "Error: Missing student information."
    if student_id in students:
        return False, "Error: Student already registered (duplicate ID)."
    students[student_id] = {"name": name, "email": email, "courses": []}
    return True, "Student registered successfully."

# Demonstration
print(register_student("26015486", "Victor", "victor@gmail.com"))
print(register_student("26015486", "Duplicate Attempt", "x@gmail.com"))
print(register_student("", "Missing ID", "y@gmail.com"))

(True, 'Student registered successfully.')
(False, 'Error: Student already registered (duplicate ID).')
(False, 'Error: Missing student information.')


## (b) Course Allocation Using Conditional Statements

A student is allocated to a course only if: (1) the student is already registered, (2) the course exists, (3) the student is not already enrolled in it, and (4) the course has not reached capacity. Each rule is enforced with a guard clause.

In [2]:
courses = {
    "Algorithms": {"capacity": 2, "students": []},
    "Databases": {"capacity": 1, "students": []},
}

def allocate_course(student_id, course_name):
    """Allocate a registered student to a course. Returns (success: bool, message: str)."""
    if student_id not in students:
        return False, "Error: Student is not registered."
    if course_name not in courses:
        return False, "Error: Course does not exist."
    course = courses[course_name]
    if student_id in course["students"]:
        return False, "Error: Student already enrolled in this course."
    if len(course["students"]) >= course["capacity"]:
        return False, "Error: Course is full."
    course["students"].append(student_id)
    students[student_id]["courses"].append(course_name)
    return True, "Course allocated successfully."

# Demonstration
print(allocate_course("26015486", "Algorithms"))
print(allocate_course("99999999", "Algorithms"))   # not registered
print(allocate_course("26015486", "Physics"))       # course doesn't exist

(True, 'Course allocated successfully.')
(False, 'Error: Student is not registered.')
(False, 'Error: Course does not exist.')


## (c) Pseudocode for the Registration Process

```text
START

INPUT student_id, name, email

IF student_id is empty OR name is empty OR email is empty THEN
    DISPLAY "Missing Information"
    STOP
END IF

IF student_id already exists in students THEN
    DISPLAY "Duplicate Registration"
    STOP
END IF

STORE student record in students

DISPLAY "Registration Successful"

END
```

```text
COURSE ALLOCATION

START

INPUT student_id, course_name

IF student_id NOT IN students THEN
    DISPLAY "Student Not Registered"
    STOP
END IF

IF course_name NOT IN courses THEN
    DISPLAY "Course Does Not Exist"
    STOP
END IF

IF student_id already in course's student list THEN
    DISPLAY "Already Enrolled"
    STOP
END IF

IF number of enrolled students >= course capacity THEN
    DISPLAY "Course Full"
    STOP
END IF

ADD student_id to course
DISPLAY "Course Allocated Successfully"

END
```

## (d) Constraint Validation

The system enforces four invariants:
1. **No duplicate student IDs**
2. **No missing information** (ID, name, email all required)
3. **No duplicate course enrollment** for the same student
4. **Capacity limits are respected**

Below, a standalone validator is implemented and then a full automated test suite proves every rule holds — including edge cases the original hand-written draft missed (allocating a course to a student who was never registered, and re-enrolling a student in the same course twice).

In [3]:
def validate_registration(student_id, name, email):
    """Pure validation check, independent of mutation."""
    if not student_id or not name or not email:
        return False, "Missing data"
    if student_id in students:
        return False, "Duplicate ID"
    return True, "Valid Registration"

print(validate_registration("26015490", "New Student", "new@gmail.com"))
print(validate_registration("26015486", "Victor", "victor@gmail.com"))  # duplicate
print(validate_registration("", "No ID", "noid@gmail.com"))             # missing

(True, 'Valid Registration')
(False, 'Duplicate ID')
(False, 'Missing data')


In [4]:
# ---- Automated test suite (constraint validation proof) ----
test_students, test_courses = {}, {"Algorithms": {"capacity": 2, "students": []}}

def reg(sid, name, email):
    if not sid or not name or not email:
        return False, "Error: Missing student information."
    if sid in test_students:
        return False, "Error: Student already registered (duplicate ID)."
    test_students[sid] = {"name": name, "email": email, "courses": []}
    return True, "OK"

def alloc(sid, cname):
    if sid not in test_students:
        return False, "Error: Student is not registered."
    if cname not in test_courses:
        return False, "Error: Course does not exist."
    c = test_courses[cname]
    if sid in c["students"]:
        return False, "Error: Student already enrolled in this course."
    if len(c["students"]) >= c["capacity"]:
        return False, "Error: Course is full."
    c["students"].append(sid)
    test_students[sid]["courses"].append(cname)
    return True, "OK"

assert reg("S1", "Alice", "a@x.com")[0] is True
assert reg("S1", "Alice2", "a2@x.com")[0] is False, "duplicate ID must fail"
assert reg("", "NoID", "n@x.com")[0] is False, "missing data must fail"
assert alloc("S1", "Algorithms")[0] is True
assert alloc("S1", "Algorithms")[0] is False, "double enrollment must fail"
reg("S2", "Bob", "b@x.com"); alloc("S2", "Algorithms")
reg("S3", "Carol", "c@x.com")
assert alloc("S3", "Algorithms")[0] is False, "capacity limit must be enforced"
assert alloc("S999", "Algorithms")[0] is False, "unregistered student must be rejected"
assert alloc("S1", "Quantum")[0] is False, "nonexistent course must be rejected"

print("ALL CONSTRAINT VALIDATION TESTS PASSED ✔")

ALL CONSTRAINT VALIDATION TESTS PASSED ✔


### Complexity Analysis

| Operation | Complexity | Reason |
|---|---|---|
| Duplicate Check | O(1) | Dictionary key lookup (average case) |
| Insertion | O(1) | Dictionary assignment |
| Course Allocation Validation | O(1) | List membership check on a small bounded course roster* |
| Overall | **O(1)** amortized per registration/allocation call | |

\*Technically `student_id in course["students"]` is O(k) where k = students already in that course; since k ≤ capacity (a constant for a given course), this is O(1) with respect to the total student population N.

**Conclusion:** The registration and allocation algorithms run in constant time per call relative to the total number of students, making the design scalable for large universities.